<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 14. DQN — Playing Atari with Deep Reinforcement Learning

- **원 논문:** [Playing Atari with Deep Reinforcement Learning](https://arxiv.org/abs/1312.5602)
- **저자 / 발표:** Volodymyr Mnih et al. · arXiv preprint (2013)
- **난이도 / 예상 시간:** 중급 · 약 65분
- **선수 지식:** PyTorch 기본, 강화학습의 state/action/reward, Bellman equation

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

외부 Gym 없이 5-state LineWorld에서 replay memory, epsilon-greedy 행동, 고정된 Bellman target과 Q-network MSE update를 구현해 greedy 정책이 오른쪽 보상에 도달하게 한다.

**원 논문과 다른 것**

논문은 Atari 화면 4장을 convolutional Q-network에 넣어 수백만 frame을 학습한다. 여기서는 one-hot 상태 5개와 2층 MLP로 Algorithm 1의 학습 역학만 재현한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2 Background, Eq. (1) | `bellman_targets`의 terminal-aware TD target | done 상태에는 bootstrap 값이 더해지지 않는지 검사 |
| §2, Eq. (2)–(3) | `QNetwork`가 선택한 action의 Q와 고정 target 사이 오차를 최소화 | Q-value shape, finite loss와 gradient를 검사 |
| §4 Deep Reinforcement Learning, Algorithm 1 | `ReplayBuffer`와 `epsilon_greedy`의 수집·표본화·탐험 경로 | buffer batch 계약과 학습 뒤 greedy return을 검사 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
y=r+\gamma(1-d)\max_{a'}Q_{\theta^-}(s',a'),\qquad
\mathcal{L}(\theta)=\mathbb{E}[(y-Q_\theta(s,a))^2]
$$

- $\theta$는 online network, $\theta^-$는 고정된 target network, $d$는 terminal flag입니다.
- replay buffer는 상관된 transition을 섞고 target network는 bootstrapping target을 느리게 변화시킵니다.
- transition 저장, batch sampling, TD target, optimization, target sync를 명시적으로 나눕니다.
- state batch $[B,D_s]$에서 action value $[B,A]$를 만들고 gather로 $Q(s,a):[B]$를 선택합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2 Background, Eq. (1)
- **노트북 구현:** `bellman_targets`의 terminal-aware TD target
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** done 상태에는 bootstrap 값이 더해지지 않는지 검사를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 논문은 Atari 화면 4장을 convolutional Q-network에 넣어 수백만 frame을 학습한다. 여기서는 one-hot 상태 5개와 2층 MLP로 Algorithm 1의 학습 역학만 재현한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. Atari DQN을 LineWorld로 축소하기

논문 §2 Eq. (1)의 Bellman 관계에서 한 step target은

$$y=r+\gamma(1-\text{done})\max_{a'}Q(s',a')$$

입니다. §2 Eq. (2)는 $(y-Q(s,a;\theta))^2$을 최소화하고, §4 Algorithm 1은
상관된 연속 경험 대신 replay memory에서 무작위 전이를 뽑습니다. 여기서는 상태 0과
4가 terminal인 다섯 칸 환경을 사용합니다. 4에 도착하면 +1, 0이면 -1입니다.

**성공 기준:** 학습 뒤 greedy agent의 평균 return이 0.8보다 크고, 시작 상태에서
`Q(right) > Q(left)`여야 합니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from collections import deque
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(140)
np.random.seed(140)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# Python environment steps stay on CPU; AI_LAB_DEVICE=cpu can avoid
# accelerator overhead for this very small neural update.


class LineWorld:
    n_states = 5
    n_actions = 2  # 0=left, 1=right

    def reset(self):
        self.state = 2
        self.steps = 0
        return self.state

    def step(self, action):
        self.steps += 1
        self.state = int(np.clip(self.state + (-1 if action == 0 else 1), 0, 4))
        done = self.state in (0, 4) or self.steps >= 8
        reward = 1.0 if self.state == 4 else (-1.0 if self.state == 0 else -0.02)
        return self.state, reward, done


def encode_states(states):
    states = torch.as_tensor(states, dtype=torch.long)
    return F.one_hot(states, num_classes=LineWorld.n_states).float().to(DEVICE)


env = LineWorld()
print(ACCELERATOR.summary())
print("LineWorld and replay storage stay on CPU; neural batches use DEVICE.")
assert env.reset() == 2
assert encode_states([0, 2, 4]).shape == (3, 5)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §4 Deep Reinforcement Learning, Algorithm 1
- **이 셀의 책임:** `ReplayBuffer`와 `epsilon_greedy`의 수집·표본화·탐험 경로
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** buffer batch 계약과 학습 뒤 greedy return을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: 고정 용량 원형 buffer를 구현하세요.
# TODO: replacement 없이 index를 뽑아 5개의 tensor를 반환하세요.
class ReplayBuffer:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, capacity):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def __len__(self):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def push(self, state, action, reward, next_state, done):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def sample(self, batch_size, rng):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


probe_buffer = ReplayBuffer(3)
for i in range(5):
    probe_buffer.push(i % 4, i % 2, float(i), (i + 1) % 5, False)
assert len(probe_buffer) == 3


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2 Background, Eq. (1) / §2, Eq. (2)–(3)
- **이 셀의 책임:** `bellman_targets`의 terminal-aware TD target / `QNetwork`가 선택한 action의 Q와 고정 target 사이 오차를 최소화
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** done 상태에는 bootstrap 값이 더해지지 않는지 검사 / Q-value shape, finite loss와 gradient를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: 5 -> 24 -> 2 action-value MLP를 정의하세요.
# TODO: terminal mask와 max를 사용하고 target graph를 detach하세요.
# TODO: online parameter를 target으로 복사하고 freeze하세요.
# TODO: 두 network 사이 parameter L2 distance를 계산하세요.
class QNetwork(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, encoded_state):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def bellman_targets(rewards, dones, next_q_values, gamma=0.95):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def hard_sync_target(online_network, target_network):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def parameter_distance(first_network, second_network):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


q_network = QNetwork().to(DEVICE)
target_network = QNetwork().to(DEVICE)
hard_sync_target(q_network, target_network)
probe_next_q = torch.tensor([[2.0, 3.0], [8.0, 9.0]], requires_grad=True)
probe_target = bellman_targets(
    torch.tensor([1.0, -1.0]), torch.tensor([0.0, 1.0]), probe_next_q
)
assert torch.allclose(probe_target, torch.tensor([3.85, -1.0]))
assert not probe_target.requires_grad


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §2 Background, Eq. (1) / §2, Eq. (2)–(3) / §4 Deep Reinforcement Learning, Algorithm 1
- **이 셀의 책임:** `bellman_targets`의 terminal-aware TD target / `QNetwork`가 선택한 action의 Q와 고정 target 사이 오차를 최소화 / `ReplayBuffer`와 `epsilon_greedy`의 수집·표본화·탐험 경로
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** done 상태에는 bootstrap 값이 더해지지 않는지 검사 / Q-value shape, finite loss와 gradient를 검사 / buffer batch 계약과 학습 뒤 greedy return을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: epsilon 확률은 무작위 action, 아니면 argmax Q를 반환하세요.
# TODO: Algorithm 1을 180 episode 실행하세요. replay가 32개 이상이면 MSE update합니다.
def epsilon_greedy(network, state, epsilon, rng):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


replay = ReplayBuffer(capacity=400)
optimizer = torch.optim.Adam(q_network.parameters(), lr=0.008)
rng = np.random.default_rng(142)
returns, losses = ([], [])
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §4 Deep Reinforcement Learning, Algorithm 1
- **이 셀의 책임:** `ReplayBuffer`와 `epsilon_greedy`의 수집·표본화·탐험 경로
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** buffer batch 계약과 학습 뒤 greedy return을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: epsilon=0 정책을 40회 평가하고 state별 Q와 학습곡선을 시각화하세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

## 7. 해석 TODO

1. replay에서 연속 전이가 아니라 무작위 전이를 뽑는 논문의 이유 두 가지를 쓰세요.
2. TD target을 `detach`하지 않으면 Eq. (2)와 어떤 점이 달라지나요?
3. Atari 원 재현과 이 축소판 사이 가장 큰 representation 차이를 적으세요.

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.